# 01 — Exploratory Data Analysis

**Catalog Campaign Profitability & Customer Targeting Analysis**

## Purpose

Before any model is built, establish three things:

1. Whether the data is fit to support a financial recommendation (data quality).
2. Which customer characteristics actually move average sale amount, and by how much in dollars.
3. Whether the 250 campaign prospects resemble the customers the model will learn from.

Every chart in this notebook answers a business question. Anything that would be
decoration has been left out.

**Business questions addressed:** BR-005 (what drives higher sales), BR-011 (data quality).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.append(str(Path.cwd().parent))
from src import data_cleaning as dc

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 12
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

customers = dc.clean_customers(dc.load_customers())
mailing = dc.clean_mailing_list(dc.load_mailing_list())

print(f"Historical customers (training): {customers.shape[0]:,} rows x {customers.shape[1]} columns")
print(f"Campaign prospects (scoring):    {mailing.shape[0]:,} rows x {mailing.shape[1]} columns")
customers.head()

## 1. Data quality

Findings are produced by code rather than asserted. The full write-up is in
[`docs/data_quality.md`](../docs/data_quality.md).

In [ ]:
qc_customers = dc.check_data_quality(customers, "p1-customers")
qc_mailing = dc.check_data_quality(mailing, "p1-mailinglist")

quality = pd.DataFrame([
    {"Check": "Rows", "Customers": qc_customers["rows"], "Mailing list": qc_mailing["rows"]},
    {"Check": "Total missing values", "Customers": qc_customers["total_missing"], "Mailing list": qc_mailing["total_missing"]},
    {"Check": "Duplicate rows", "Customers": qc_customers["duplicate_rows"], "Mailing list": qc_mailing["duplicate_rows"]},
    {"Check": "Duplicate Customer_IDs", "Customers": qc_customers["duplicate_ids"], "Mailing list": qc_mailing["duplicate_ids"]},
    {"Check": "Unexpected segment levels", "Customers": len(qc_customers["unexpected_segments"]), "Mailing list": len(qc_mailing["unexpected_segments"])},
    {"Check": "Negative product counts", "Customers": qc_customers["negative_products"], "Mailing list": qc_mailing["negative_products"]},
])
quality

In [ ]:
# Probability integrity on the mailing list: Score_Yes + Score_No should equal 1 exactly
print(f"Score_Yes out of [0,1] range: {qc_mailing['Score_Yes_out_of_range']}")
print(f"Max deviation of (Score_Yes + Score_No) from 1.0: {qc_mailing['max_score_sum_deviation']:.2e}")
print(f"Non-positive Avg_Sale_Amount in training data: {qc_customers['non_positive_target']}")
print(f"Avg_Sale_Amount range: ${qc_customers['target_min']:,.2f} to ${qc_customers['target_max']:,.2f}")

**Data-quality verdict.** No missing values, no duplicates, no invalid ranges and
no unexpected category labels in either file. The issues that do exist are
definitional rather than corruption:

| Issue | Consequence |
|---|---|
| `ZIP` and `Store_Number` stored as integers | Numeric labels, not quantities. Cast and excluded from modelling. |
| `#_Years_as_Customer` is `int` in one file, `float` on a different scale in the other | The two files do not appear to measure the same thing. Excluded (constraint C-05). |
| `Responded_to_Last_Catalog` absent from the mailing list | Cannot be a model feature — the model would be unscoreable on the campaign population. |

The data is fit for purpose.

## 2. What is the target variable actually like?

`Avg_Sale_Amount` is what the model must predict, and its shape determines how
much confidence any single prediction deserves.

In [ ]:
target = customers["Avg_Sale_Amount"]
outlier_mask = dc.flag_outliers_iqr(target)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(target, bins=50, color="#2b6cb0", edgecolor="white")
axes[0].axvline(target.mean(), color="#c53030", linestyle="--", label=f"Mean ${target.mean():,.0f}")
axes[0].axvline(target.median(), color="#2f855a", linestyle="--", label=f"Median ${target.median():,.0f}")
axes[0].set_title("Distribution of average sale amount")
axes[0].set_xlabel("Average sale amount ($)"); axes[0].set_ylabel("Customers"); axes[0].legend()

axes[1].boxplot(target, vert=False, widths=0.6, patch_artist=True,
                boxprops=dict(facecolor="#bee3f8"), medianprops=dict(color="#c53030"))
axes[1].set_title(f"Spread and outliers ({outlier_mask.sum()} beyond the 1.5x IQR fence)")
axes[1].set_xlabel("Average sale amount ($)"); axes[1].set_yticks([])
plt.tight_layout(); plt.show()

print(target.describe().to_string())
print(f"\nOutliers beyond 1.5x IQR: {outlier_mask.sum()} of {len(target)} ({outlier_mask.mean():.1%})")

### Business interpretation

The distribution is strongly right-skewed: the mean ($399.77) sits well above the
median ($281.32), and the range runs from $1.22 to $2,963.49. A small group of
high-value customers accounts for a disproportionate share of revenue.

**Why this matters for the campaign.** An "average customer" is a misleading
planning unit. Treating every recipient as worth the mean would badly
under-value the top of the list and over-value the bottom — which is precisely
the case for scoring customers individually rather than mailing on segment
intuition.

**Decision on the 58 outliers: retain them.** They are not data errors. They are
genuine high-value customers, concentrated in the segment the campaign most wants
to reach. Removing them would bias the model against its most profitable targets.
The cost of retaining them is that ordinary least squares under-predicts at the
top of the range, which is documented in the residual analysis in notebook 02.

## 3. Does the number of products purchased drive sale amount?

The most obvious candidate predictor, and the only continuous one available.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
for seg, colour in zip(sorted(customers["Customer_Segment"].unique()),
                       ["#2b6cb0", "#dd6b20", "#2f855a", "#805ad5"]):
    sub = customers[customers["Customer_Segment"] == seg]
    ax.scatter(sub["Avg_Num_Products_Purchased"], sub["Avg_Sale_Amount"],
               alpha=0.45, s=18, label=seg, color=colour)

r = customers["Avg_Num_Products_Purchased"].corr(customers["Avg_Sale_Amount"])
ax.set_title(f"Average sale amount vs products purchased (r = {r:.2f})")
ax.set_xlabel("Average number of products purchased")
ax.set_ylabel("Average sale amount ($)")
ax.legend(title="Customer segment", fontsize=8)
plt.tight_layout(); plt.show()

print(f"Correlation with average sale amount: {r:.3f}")

### Business interpretation

Products purchased is the strongest single driver available, correlating at
**r = 0.86** with average sale amount. The relationship is close to linear, which
supports a linear model rather than something more flexible.

The colouring reveals the second pattern: **segments sit in distinct bands**.
Customers with the same basket size spend materially different amounts depending
on their relationship with the company. Segment is not a proxy for basket size —
it carries independent information, which is why both variables belong in the model.

**Caveat worth stating plainly.** Both fields are averages over the same purchase
history. Part of the correlation is arithmetic — customers who buy more items
naturally have larger baskets — rather than a discovered behavioural driver. This
does not invalidate the model for forecasting, but it does mean the relationship
should not be read as "persuading a customer to add an item is worth $67".

## 4. Which segments are worth the most?

Segment is the unit marketing actually plans and buys against.

In [ ]:
seg_stats = (customers.groupby("Customer_Segment")["Avg_Sale_Amount"]
             .agg(Customers="count", Mean="mean", Median="median", Std="std")
             .sort_values("Mean", ascending=False))
seg_stats["Share_of_customers"] = seg_stats["Customers"] / len(customers)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
order = seg_stats.index.tolist()
sns.barplot(data=customers, y="Customer_Segment", x="Avg_Sale_Amount", order=order,
            errorbar=("ci", 95), color="#2b6cb0", ax=axes[0])
axes[0].set_title("Mean average sale amount by segment (95% CI)")
axes[0].set_xlabel("Average sale amount ($)"); axes[0].set_ylabel("")

sns.boxplot(data=customers, y="Customer_Segment", x="Avg_Sale_Amount", order=order,
            color="#bee3f8", ax=axes[1])
axes[1].set_title("Full spread within each segment")
axes[1].set_xlabel("Average sale amount ($)"); axes[1].set_ylabel("")
plt.tight_layout(); plt.show()

seg_stats

### Business interpretation

The segments separate sharply, and the ordering is commercially intuitive:

| Segment | Mean sale | Read |
|---|---|---|
| Loyalty Club and Credit Card | $1,074 | Deepest relationship, highest spend |
| Credit Card Only | $683 | Strong single-product relationship |
| Loyalty Club Only | $396 | Engaged but lower value |
| Store Mailing List | $157 | Weakest relationship, lowest spend |

A `Loyalty Club and Credit Card` customer is worth roughly **seven times** a
`Store Mailing List` customer. The 95% confidence intervals do not overlap
between adjacent segments, so these are real differences rather than noise.

**Campaign implication.** Segment mix drives campaign value more than list size
does. A list of 100 dual-relationship customers is worth more than 400 store-list
names.

## 5. How does the campaign population compare to the training population?

The model learns from historical customers but is applied to the 250 prospects.
If those populations differ materially, the forecast leans on parts of the model
estimated from thinner evidence.

In [ ]:
mix = pd.DataFrame({
    "Training share": customers["Customer_Segment"].value_counts(normalize=True),
    "Mailing list share": mailing["Customer_Segment"].value_counts(normalize=True),
}).fillna(0).sort_values("Mailing list share", ascending=False)

ax = mix.plot(kind="barh", figsize=(10, 4.5), color=["#a0aec0", "#2b6cb0"])
ax.set_title("Segment mix: training population vs campaign prospects")
ax.set_xlabel("Share of population"); ax.set_ylabel("")
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
plt.tight_layout(); plt.show()

(mix * 100).round(1)

### Business interpretation

The two populations are **not** alike. `Store Mailing List` is 47% of the
training data but only 8% of the campaign list, while `Loyalty Club Only` rises
from 24% to 49% and `Credit Card Only` from 21% to 33%.

**This is not an error** — a curated prospect list should skew towards
higher-value relationships. But it has two consequences that must be carried into
the recommendation:

1. The campaign forecast depends heavily on segments estimated from a smaller
   share of the training data. The `Loyalty Club and Credit Card` coefficient
   rests on just 194 training records.
2. Because the list is skewed towards richer segments, the average predicted sale
   amount for prospects ($553) is well above the historical average ($400). That
   is a real effect of list composition, not model optimism — but it is worth
   stating explicitly so nobody reads it as the model inflating its own forecast.

Registered as constraint **C-04** and risk **R-11**. Segment-level performance
should be reported separately after launch so any segment where the model misses
is identified rather than averaged away.

## 6. Does responding to the last catalog predict higher spend?

An obvious hypothesis: past responders are the best targets.

In [ ]:
resp = (customers.groupby("Responded_to_Last_Catalog")["Avg_Sale_Amount"]
        .agg(Customers="count", Mean="mean", Median="median"))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.boxplot(data=customers, x="Responded_to_Last_Catalog", y="Avg_Sale_Amount",
            order=["No", "Yes"], palette=["#a0aec0", "#2b6cb0"], ax=axes[0])
axes[0].set_title("Average sale amount by prior catalog response")
axes[0].set_xlabel("Responded to last catalog"); axes[0].set_ylabel("Average sale amount ($)")

cross = pd.crosstab(customers["Customer_Segment"], customers["Responded_to_Last_Catalog"],
                    normalize="index")
cross.plot(kind="barh", stacked=True, ax=axes[1], color=["#a0aec0", "#2b6cb0"])
axes[1].set_title("Prior response rate within each segment")
axes[1].set_xlabel("Share of segment"); axes[1].set_ylabel("")
axes[1].legend(title="Responded", loc="lower right")
plt.tight_layout(); plt.show()

resp

### Business interpretation — a counter-intuitive result worth pausing on

Customers who responded to the last catalog have a **lower** average sale amount
($156) than those who did not ($419). The naive reading — "past responders are
our best targets" — is contradicted by the data.

The right-hand chart explains why. Prior response is concentrated in the
`Store Mailing List` segment, which is also the lowest-spending segment. This is
a textbook case of **confounding**: the apparent effect of response is really the
effect of segment.

**Two things follow.**

1. *Analytically*: this field would be misleading as a standalone targeting rule.
   Anyone filtering the mail file to "previous responders" would systematically
   select the lowest-value customers.
2. *Practically*: it does not matter for this campaign, because the field is
   absent from the mailing list and is excluded under BRule-06 anyway.

It is included here because it is the kind of finding a stakeholder is likely to
propose in the room, and having the answer ready is more useful than the chart.

## 7. Does tenure matter?

In [ ]:
num_cols = ["Avg_Sale_Amount", "Avg_Num_Products_Purchased", "Years_as_Customer"]
corr = customers[num_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True, ax=axes[0], cbar_kws={"shrink": 0.8})
axes[0].set_title("Correlation between numeric variables")

axes[1].scatter(customers["Years_as_Customer"], customers["Avg_Sale_Amount"],
                alpha=0.3, s=15, color="#805ad5")
axes[1].set_title(f"Sale amount vs tenure (r = {corr.loc['Avg_Sale_Amount','Years_as_Customer']:.2f})")
axes[1].set_xlabel("Years as customer"); axes[1].set_ylabel("Average sale amount ($)")
plt.tight_layout(); plt.show()

corr

### Business interpretation

Tenure has essentially **no relationship** with average sale amount (r = 0.03).
Long-standing customers do not spend more per purchase than recent ones.

This is a useful negative finding. It means loyalty-length is not a targeting
criterion, and it removes a variable that a stakeholder might otherwise assume
should be in the model. Combined with the definitional inconsistency between the
two files (constraint C-05), tenure is excluded.

Note also that products purchased and tenure are effectively uncorrelated
(r = 0.04), so there is no multicollinearity concern in the retained feature set.

## 8. What does the campaign population look like?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(mailing["Score_Yes"], bins=30, color="#2f855a", edgecolor="white")
axes[0].axvline(mailing["Score_Yes"].mean(), color="#c53030", linestyle="--",
                label=f"Mean {mailing['Score_Yes'].mean():.1%}")
axes[0].set_title("Supplied response probability across the 250 prospects")
axes[0].set_xlabel("P(response)"); axes[0].set_ylabel("Prospects"); axes[0].legend()

sns.boxplot(data=mailing, y="Customer_Segment", x="Score_Yes", color="#c6f6d5", ax=axes[1])
axes[1].set_title("Response probability by segment")
axes[1].set_xlabel("P(response)"); axes[1].set_ylabel("")
plt.tight_layout(); plt.show()

print(mailing["Score_Yes"].describe().to_string())

### Business interpretation

The supplied response probabilities average **34.1%**, ranging from 18.6% to
effectively 100%. The distribution is right-skewed — most prospects cluster in
the 20-30% band with a tail of high-probability names.

Crucially, response probability is **roughly flat across segments** (means of
33.7% to 35.9%). Segment predicts *how much* a customer spends; it does not
predict *whether* they respond. The two inputs to expected revenue are therefore
largely independent, which is exactly what makes ranking on the product of the
two worthwhile — the highest-value targets are not simply the highest-probability
ones.

**Important caveat.** `Score_Yes` arrives with the data. The model that produced
it is not supplied, so its methodology and calibration cannot be validated here.
It is used as given (assumption **A-05**) and stress-tested across a 60%-120%
band in notebook 04 rather than taken on trust.

## Summary of EDA findings

| # | Finding | Consequence for the project |
|---|---|---|
| 1 | Data is clean: no missing values, duplicates or invalid ranges | Analysis can proceed without imputation |
| 2 | Sale amount is right-skewed with 58 genuine high-value outliers | Retained; model under-predicts at the top of the range |
| 3 | Products purchased correlates at r = 0.86 with sale amount | Primary numeric predictor |
| 4 | Segments differ sharply, from $157 to $1,074 mean sale | Second predictor; carries information independent of basket size |
| 5 | Tenure is uncorrelated with spend (r = 0.03) | Excluded from the model |
| 6 | Prior catalog response is confounded with segment | Excluded; would mislead as a targeting rule |
| 7 | Campaign list skews towards higher-value segments than the training data | Constraint C-04, risk R-11; report segment performance separately post-launch |
| 8 | Response probability is flat across segments | Value ranking and probability ranking are different things; use the product |

**Feature set for modelling:** `Avg_Num_Products_Purchased` and
`Customer_Segment`. Proceed to [`02_predictive_model.ipynb`](02_predictive_model.ipynb).